In [0]:
from pyspark.sql import functions as F

# Bronze Volume Path
bronze_path = "/Volumes/tt_hc_adb_ws/bronze/bronze_volume"

# Reading Hospital A departments data
df_hosa = spark.read.parquet(f"{bronze_path}/hosa/departments")

# Reading Hospital B departments data
df_hosb = spark.read.parquet(f"{bronze_path}/hosb/departments")

# Union the two DataFrames
df_merged = df_hosa.unionByName(df_hosb)

# Create SRC_Dept_id and Dept_id
df_merged = (
    df_merged
    .withColumn("SRC_Dept_id", F.col("deptid"))
    .withColumn(
        "Dept_id",
        F.concat(F.col("deptid"), F.lit("-"), F.col("datasource"))
    )
    .drop("deptid")
)

# Create Temp View
df_merged.createOrReplaceTempView("departments")

display(df_merged)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS tt_hc_adb_ws.silver.departments (
Dept_Id string,
SRC_Dept_Id string,
Name string,
datasource string,
is_quarantined boolean
)
USING DELTA;

In [0]:
%sql 
truncate table tt_hc_adb_ws.silver.departments

In [0]:
%sql
insert into tt_hc_adb_ws.silver.departments
SELECT 
Dept_Id,
SRC_Dept_Id,
Name,
Datasource,
    CASE 
        WHEN SRC_Dept_Id IS NULL OR Name IS NULL THEN TRUE
        ELSE FALSE
    END AS is_quarantined
FROM departments

In [0]:
%sql
select * from tt_hc_adb_ws.silver.departments